In [1]:
# Fine tune modernbert base embed model with squad data to create a fine tuned embedding model
# which we will compare with the default sentence transformer model

import pandas as pd
# CSVs with the format question,context, answers
train_df_all = pd.read_csv("../data/squad_v2/train.csv")
val_df_all = pd.read_csv("../data/squad_v2/validation.csv")
len(train_df_all), len(val_df_all)

(130319, 11873)

In [2]:
# Filter those rows which are answerable from squad v2 dataset, so as to take only those query and context pair for proper training
answer_col = "answers" 
pattern = r"""['"]text['"]\s*:\s*array\(\s*\[\s*['"]"""
train_df = train_df_all[
    train_df_all[answer_col].str.contains(pattern, regex=True, na=False)
].copy()
val_df = val_df_all[
    val_df_all[answer_col].str.contains(pattern, regex=True, na=False)
].copy()
len(train_df), len(val_df)

(86821, 5928)

In [3]:
# Sample N records
train_df_all.head(1000)

,id,title,context,question,answers
0,56be85543aeaaa14008c9063,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce start becoming popular?,"{'text': array(['in the late 1990s'], dtype=ob..."
1,56be85543aeaaa14008c9065,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,What areas did Beyonce compete in when she was...,"{'text': array(['singing and dancing'], dtype=..."
2,56be85543aeaaa14008c9066,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce leave Destiny's Child and bec...,"{'text': array(['2003'], dtype=object), 'answe..."
3,56bf6b0f3aeaaa14008c9601,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In what city and state did Beyonce grow up?,"{'text': array(['Houston, Texas'], dtype=objec..."
4,56bf6b0f3aeaaa14008c9602,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In which decade did Beyonce become famous?,"{'text': array(['late 1990s'], dtype=object), ..."
...,...,...,...,...,...
995,56d319d459d6e41400146241,Frédéric_Chopin,"In Paris, Chopin encountered artists and other...",What position did the poet who Chopin knew in ...,{'text': array(['principal of the Polish Liter...
996,56cbf5106d243a140015ee18,Frédéric_Chopin,Two Polish friends in Paris were also to play ...,Which friend of Frédéric failed to achieve suc...,"{'text': array(['Julian Fontana'], dtype=objec..."
997,56cbf5106d243a140015ee19,Frédéric_Chopin,Two Polish friends in Paris were also to play ...,Who was Frédéric's trusted adviser while in Pa...,"{'text': array(['Albert Grzymała'], dtype=obje..."
998,56cbf5106d243a140015ee1a,Frédéric_Chopin,Two Polish friends in Paris were also to play ...,What familial role was Albert Grzymała compare...,"{'text': array(['elder brother'], dtype=object..."


In [4]:
# Taking 50 samples each in train and validation
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df[["question", "context"]].head(50).rename(columns={'question': 'query'}), preserve_index=False)
val_dataset =  Dataset.from_pandas(val_df[["question", "context"]].head(50).rename(columns={'question': 'query'}), preserve_index=False)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(val_dataset)}")
train_dataset, val_dataset


H:\pythonProject\natural_language_processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train dataset size: 50
Test dataset size: 50


(Dataset({
     features: ['query', 'context'],
     num_rows: 50
 }),
 Dataset({
     features: ['query', 'context'],
     num_rows: 50
 }))

In [5]:
# Fine Tuning 
import torch
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers import SentenceTransformer

# locally saved pretrained model
model_name = "../model/modernbert-embed-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
# For query & context pair, multiple negative ranking loss is desired
loss = MultipleNegativesRankingLoss(model)


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 136.83it/s]


In [6]:
from sentence_transformers import SentenceTransformerTrainingArguments , SentenceTransformerTrainer

args = SentenceTransformerTrainingArguments(
    output_dir="../output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=False,
    eval_strategy="steps",
    eval_steps=0.125,
    save_strategy="steps",
    save_steps=0,
    save_total_limit=2,
    logging_steps=1,
    report_to="none"
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss,
)



trainer.train()

H:\pythonProject\natural_language_processing\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
10,0.693148,0.261636
20,0.126890,0.252651
30,0.110988,0.264804
40,0.011354,0.265772
50,0.009395,0.266358
60,0.000199,0.265857
70,0.021112,0.266171
75,0.002141,0.266283


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.18s/it]


TrainOutput(global_step=75, training_loss=0.27418203989664713, metrics={'train_runtime': 2266.2888, 'train_samples_per_second': 0.066, 'train_steps_per_second': 0.033, 'total_flos': 0.0, 'train_loss': 0.27418203989664713, 'epoch': 3.0})

In [7]:
search_query_str = "search_query: "
search_document_str= "search_document: "
query_list = ["what is the maturity date for abc inc?"]
document_list = ["The date at which india got independence is 15th august 1947", 
                 "The maturity date is February 26th 2019", 
                 "Abc inc's maturity date is 2nd June 2021"]

query_list_final = [search_query_str + x for x in query_list]
document_list_final = [search_document_str + x for x in document_list]

# Evaluate which is the best model, with fine tuning the domain specific fine tuned model should be better
fine_tuned_model = "../output/checkpoint-75"
base_model = "../model/modernbert-embed-base"
generic_embedding_model = "../model/all-MiniLM-L6-v2"
all_models = [fine_tuned_model, base_model, generic_embedding_model]

for m in all_models:
    print("model", m)
    model = SentenceTransformer(m, device=device)
    query_embed = model.encode(query_list_final)
    doc_embed = model.encode(document_list_final)
    print(query_embed.shape, doc_embed.shape)
    similarity = model.similarity(query_embed, doc_embed)
    print("Similarity", similarity)

model ../output/checkpoint-75


Loading weights: 100%|██████████| 134/134 [00:01<00:00, 93.12it/s]


(1, 768) (3, 768)
Similarity tensor([[0.2523, 0.5978, 0.8479]])
model ../model/modernbert-embed-base


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 136.16it/s]


(1, 768) (3, 768)
Similarity tensor([[0.2363, 0.5766, 0.8409]])
model ../model/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1161.59it/s]


(1, 384) (3, 384)
Similarity tensor([[0.1781, 0.7097, 0.8739]])
